In [ ]:
A JWT token looks like this:
    header.payload.signature
        
        1️⃣ Header (algorithm info)
        {
          "alg": "HS256",
          "typ": "JWT"
        }
        
        2️⃣ Payload (user info + expiry)
        {
          "user_id": 10,
          "role": "admin",
          "iat": 1733875795,
          "exp": 1733876695
        }
        
        3️⃣ Signature (security)

eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.
eyJ1c2VyX2lkIjoxMCwiZXhwIjoxNzM2MDczMDAwfQ.
SxAMQ7PlpMOh7VJd3T2z_Cz1tT-LNOXPBZf8VOyJ_dk

When the client sends this token, the server must check:

    🔥 How the server re-generates the signature

Server takes the first 2 parts:

   encoded_header + "." + encoded_payload

Then the server uses the same secret key:
   SECRET_KEY = "mysecret123"
And same algorithm (HS256):

   HMACSHA256(header + "." + payload, SECRET_KEY)

This produces a new signature.

    Server compares:

        new_signature == signature_from_token ?
✔ If they match → Token is valid
❌ If not → Token is fake or modified

4️⃣ Also checks:

    Timestamp
exp (expiration)

iat

nbf

In [ ]:
✅ 2. PyJWT Automatic Verification

👉 This is how you normally verify a JWT in Python:

import jwt
from django.conf import settings

def verify_jwt_pyjwt(token):
    try:
        payload = jwt.decode(
            token,
            settings.SECRET_KEY,
            algorithms=["HS256"]
        )
        return payload

    except jwt.ExpiredSignatureError:
        return "Token expired"

    except jwt.InvalidTokenError:
        return "Invalid token"


In [ ]:
PyJWT automatically checks:

✔ signature
✔ exp
✔ iat
✔ nbf

In [ ]:
JWT Payload:-

{
  "user_id": 1,
  "exp": 1738000000,
  "iat": 1737996400,
  "nbf": 1737996500
}

In [ ]:
“JWT verification checks multiple claims.
exp ensures token is not used after expiry.
iat ensures token wasn’t issued in the future.
nbf ensures token is only valid after a specific time.
And finally, the signature ensures the token wasn’t modified.
These combined ensure authenticity and validity of the JWT.”

In [ ]:
✅ 3. Django REST Framework – Custom JWT Authentication Class

👉 This is the industry-standard authentication pipeline.

    from rest_framework.authentication import BaseAuthentication
from rest_framework.exceptions import AuthenticationFailed
from django.contrib.auth import get_user_model
from .utils import decode_jwt_token
from .blacklist import is_blacklisted

User = get_user_model()

class CustomJWTAuthentication(BaseAuthentication):

    def authenticate(self, request):
        # 1. Get auth header
        auth = request.headers.get("Authorization")
        if not auth:
            return None

        if not auth.startswith("Bearer "):
            raise AuthenticationFailed("Invalid token prefix")

        # 2. Extract token
        token = auth.split(" ")[1]

        # 3. Verify JWT (signature + exp)
        payload = decode_jwt_token(token)
        if payload is None:
            raise AuthenticationFailed("Invalid or expired token")

        jti = payload.get("jti")

        # 4. Check blacklist
        if is_blacklisted(jti):
            raise AuthenticationFailed("Token is blacklisted")

        # 5. Check if user exists
        try:
            user = User.objects.get(id=payload["user_id"])
        except User.DoesNotExist:
            raise AuthenticationFailed("User not found")

        return (user, None)


✔ Verification Steps Here:

Extract token

Decode token (signature + algorithm)

Check expiration

Check blacklist

Load user

Authenticate user